# Amine Replicate Consistency Analysis

**Goal:** Characterize which amines are reliably detected as deconjugation products vs. which show inconsistent/noisy signal across the 3 experimental replicates.

**Data source:** Raw replicate-level intensities from:
- `NEW_Stage2_BAs_subs_for_heatmap_manual.csv` (12 substrate products: taurine + glycine conjugates)
- `NEW_Stage2_BAs_amines_for_heatmap_manual.csv` (82 amine products: 27 amines across bile acid cores)

Each enzyme has 3 replicates (rep1, rep2, rep3). Each product column contains the raw intensity.

## Section 1: Imports & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import re

DATA_DIR = Path("../data")
OUTPUT_DIR = Path("../outputs")
RESULTS_DIR = OUTPUT_DIR / "model_outputs" / "amine_consistency"
RESULTS_DIR.mkdir(exist_ok=True, parents=True)

# Load both heatmap CSVs
df_subs = pd.read_csv(DATA_DIR / "NEW_Stage2_BAs_subs_for_heatmap_manual.csv")
df_amines = pd.read_csv(DATA_DIR / "NEW_Stage2_BAs_amines_for_heatmap_manual.csv")

print(f"Substrates file: {df_subs.shape[0]} rows x {df_subs.shape[1]} cols ({df_subs.shape[1]-3} products)")
print(f"Amines file: {df_amines.shape[0]} rows x {df_amines.shape[1]} cols ({df_amines.shape[1]-3} products)")
print(f"\nReplicate values: {df_subs['Replicate'].unique()}")

In [ ]:
# Merge both files on (filename, Code, Replicate) to get all products in one DataFrame
df_raw = pd.merge(df_subs, df_amines, on=['filename', 'Code', 'Replicate'], how='outer')
print(f"Merged: {df_raw.shape[0]} rows x {df_raw.shape[1]} cols")

# Remove blanks
df_raw = df_raw[df_raw['Code'].notna() & (df_raw['Code'] != 'NA')].copy()
print(f"After removing blanks: {df_raw.shape[0]} rows")
print(f"Unique enzymes: {df_raw['Code'].nunique()}")
print(f"\nReplicates per enzyme:")
print(df_raw.groupby('Code')['Replicate'].count().value_counts().sort_index())

# Product columns = everything except meta columns
meta_cols = ['filename', 'Code', 'Replicate']
product_cols = [c for c in df_raw.columns if c not in meta_cols]
print(f"\nTotal product columns: {len(product_cols)}")

In [ ]:
# Parse product column names: {Hydroxyl}_{Amine}_{numericID}
# The last token (after last _) is always the numeric ID
# The first token is the hydroxyl pattern
# Everything in between is the amine name

def parse_product_col(col):
    """Parse product column name into (hydroxyl, amine, product_id)."""
    parts = col.rsplit('_', 1)  # split from right to get numeric ID
    if len(parts) != 2 or not parts[1].isdigit():
        return None, None, None
    product_id = parts[1]
    remainder = parts[0]  # {Hydroxyl}_{Amine}
    
    # Split hydroxyl from amine — hydroxyl is the first token
    # But some hydroxyl patterns contain commas: '3a,7a,12k'
    # Known hydroxyl patterns:
    known_hydroxyls = ['3a,7a,12k', '3a7a12k', '3a12k', '3a7k', '3k12a', '3k7a', 'Di', 'Mono', 'Tri']
    
    for h in sorted(known_hydroxyls, key=len, reverse=True):  # try longest first
        if remainder.startswith(h + '_'):
            amine = remainder[len(h)+1:]
            return h, amine, product_id
    
    return None, None, None

product_info = {}
for col in product_cols:
    h, a, pid = parse_product_col(col)
    if h is not None:
        product_info[col] = {'hydroxyl': h, 'amine': a, 'product_id': pid}

print(f"Parsed {len(product_info)} / {len(product_cols)} product columns")

# Unique amines and hydroxyl patterns
unique_amines = sorted(set(v['amine'] for v in product_info.values()))
unique_hydroxyls = sorted(set(v['hydroxyl'] for v in product_info.values()))
print(f"\nUnique amines ({len(unique_amines)}): {unique_amines}")
print(f"\nUnique hydroxyl patterns ({len(unique_hydroxyls)}): {unique_hydroxyls}")

## Section 2: Melt to Long Format & Compute Replicate Stats

In [ ]:
# Melt product columns into long format
df_long = df_raw.melt(
    id_vars=['Code', 'Replicate'],
    value_vars=list(product_info.keys()),
    var_name='product',
    value_name='intensity'
)

# Add parsed amine and hydroxyl
df_long['amine'] = df_long['product'].map(lambda x: product_info[x]['amine'])
df_long['hydroxyl'] = df_long['product'].map(lambda x: product_info[x]['hydroxyl'])

# Fill NaN intensities with 0
df_long['intensity'] = df_long['intensity'].fillna(0)

# Binary detection: intensity > 0
df_long['detected'] = (df_long['intensity'] > 0).astype(int)

print(f"Long format: {len(df_long)} rows")
print(f"  (enzymes x products x replicates = {df_raw['Code'].nunique()} x {len(product_info)} x 3 = {df_raw['Code'].nunique() * len(product_info) * 3})")
print(df_long.head(10))

In [ ]:
# For each (enzyme, product), compute stats across the 3 replicates
replicate_stats = df_long.groupby(['Code', 'product', 'amine', 'hydroxyl']).agg(
    n_reps=('intensity', 'count'),
    mean_intensity=('intensity', 'mean'),
    std_intensity=('intensity', 'std'),
    max_intensity=('intensity', 'max'),
    min_intensity=('intensity', 'min'),
    n_detected=('detected', 'sum'),
).reset_index()

# CV of intensity (only where mean > 0)
replicate_stats['cv'] = np.where(
    replicate_stats['mean_intensity'] > 0,
    replicate_stats['std_intensity'] / replicate_stats['mean_intensity'],
    np.nan
)

# Detection consistency categories
def detection_category(row):
    if row['n_detected'] == 0:
        return 'never_detected'      # 0/3 replicates
    elif row['n_detected'] == row['n_reps']:
        return 'always_detected'      # 3/3 replicates
    else:
        return 'inconsistent'         # 1/3 or 2/3 replicates

replicate_stats['detection'] = replicate_stats.apply(detection_category, axis=1)

print(f"Replicate stats: {len(replicate_stats)} (enzyme, product) combos")
print(f"\nDetection categories:")
print(replicate_stats['detection'].value_counts())

## Section 3: Per-Amine Detection Consistency

In [ ]:
# Aggregate detection consistency per amine (across all enzymes and products for that amine)
amine_detection = replicate_stats.groupby('amine').agg(
    n_combos=('detection', 'count'),
    n_never=('detection', lambda x: (x == 'never_detected').sum()),
    n_always=('detection', lambda x: (x == 'always_detected').sum()),
    n_inconsistent=('detection', lambda x: (x == 'inconsistent').sum()),
).reset_index()

amine_detection['pct_never'] = amine_detection['n_never'] / amine_detection['n_combos'] * 100
amine_detection['pct_always'] = amine_detection['n_always'] / amine_detection['n_combos'] * 100
amine_detection['pct_inconsistent'] = amine_detection['n_inconsistent'] / amine_detection['n_combos'] * 100
amine_detection['pct_consistent'] = (amine_detection['n_never'] + amine_detection['n_always']) / amine_detection['n_combos'] * 100

# Sort by consistency rate
amine_detection = amine_detection.sort_values('pct_consistent', ascending=True)

print("Per-amine detection consistency (sorted by % consistent):")
print(amine_detection[['amine', 'n_combos', 'pct_never', 'pct_always', 'pct_inconsistent', 'pct_consistent']].to_string(index=False))

In [ ]:
# Figure: Stacked bar chart — detection consistency per amine
fig, ax = plt.subplots(figsize=(12, 8))

df_plot = amine_detection.sort_values('pct_inconsistent', ascending=False)
y_pos = range(len(df_plot))

ax.barh(y_pos, df_plot['pct_always'], color='#2ecc71', label='Always detected (3/3)', edgecolor='white', linewidth=0.5)
ax.barh(y_pos, df_plot['pct_inconsistent'], left=df_plot['pct_always'], color='#e74c3c', label='Inconsistent (1/3 or 2/3)', edgecolor='white', linewidth=0.5)
ax.barh(y_pos, df_plot['pct_never'], left=df_plot['pct_always'] + df_plot['pct_inconsistent'], color='#bdc3c7', label='Never detected (0/3)', edgecolor='white', linewidth=0.5)

ax.set_yticks(y_pos)
ax.set_yticklabels(df_plot['amine'], fontsize=9)
ax.set_xlabel('% of (Enzyme, Product) Combinations', fontsize=12)
ax.set_title('Detection Consistency Across 3 Replicates Per Amine\n(each enzyme-product combo classified by how many replicates detected it)', fontsize=13)
ax.legend(fontsize=10, loc='lower right')
ax.set_xlim(0, 100)
ax.grid(True, alpha=0.2, axis='x')

# Add inconsistency % labels
for i, (_, row) in enumerate(df_plot.iterrows()):
    if row['pct_inconsistent'] > 2:
        ax.text(row['pct_always'] + row['pct_inconsistent']/2, i,
                f"{row['pct_inconsistent']:.0f}%", va='center', ha='center', fontsize=7, color='white', fontweight='bold')

plt.tight_layout()
plt.savefig(RESULTS_DIR / "detection_consistency_per_amine.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved: detection_consistency_per_amine.png")

## Section 4: Intensity Variability (CV) Across Replicates

In [ ]:
# For products that were detected at least once, compute CV across replicates
df_detected = replicate_stats[replicate_stats['n_detected'] > 0].copy()

print(f"Products detected at least once: {len(df_detected)} / {len(replicate_stats)}")
print(f"\nCV distribution (detected products):")
print(df_detected['cv'].describe())

# Per-amine CV stats
amine_cv = df_detected.groupby('amine').agg(
    n_detected_combos=('cv', 'count'),
    median_cv=('cv', 'median'),
    mean_cv=('cv', 'mean'),
    mean_intensity=('mean_intensity', 'median'),
).reset_index().sort_values('median_cv')

print("\nIntensity CV per amine (lower = more reproducible):")
print(amine_cv.to_string(index=False))

In [ ]:
# Figure: Box plot of CV per amine
# Only include amines that have enough detected combos
amines_with_data = amine_cv[amine_cv['n_detected_combos'] >= 10]['amine'].tolist()
df_box = df_detected[df_detected['amine'].isin(amines_with_data)].copy()

# Order by median CV
order = amine_cv[amine_cv['amine'].isin(amines_with_data)].sort_values('median_cv')['amine'].tolist()

fig, ax = plt.subplots(figsize=(14, 7))
sns.boxplot(
    data=df_box, x='amine', y='cv', order=order,
    color='steelblue', fliersize=2, ax=ax
)
ax.set_xlabel('Amine', fontsize=12)
ax.set_ylabel('Coefficient of Variation (across 3 replicates)', fontsize=12)
ax.set_title('Intensity Reproducibility Per Amine\n(lower CV = more consistent intensity across replicates)', fontsize=13)
ax.axhline(1.0, color='red', linestyle='--', alpha=0.5, label='CV = 1.0')
ax.legend(fontsize=10)
plt.xticks(rotation=45, ha='right')
ax.grid(True, alpha=0.2, axis='y')

plt.tight_layout()
plt.savefig(RESULTS_DIR / "intensity_cv_boxplot.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved: intensity_cv_boxplot.png")

## Section 5: Per-Enzyme Consistency

In [ ]:
# Which enzymes are most/least consistent across their replicates?
enzyme_consistency = replicate_stats.groupby('Code').agg(
    n_products=('detection', 'count'),
    n_always=('detection', lambda x: (x == 'always_detected').sum()),
    n_inconsistent=('detection', lambda x: (x == 'inconsistent').sum()),
    n_never=('detection', lambda x: (x == 'never_detected').sum()),
).reset_index()

enzyme_consistency['pct_inconsistent'] = enzyme_consistency['n_inconsistent'] / enzyme_consistency['n_products'] * 100
enzyme_consistency['n_total_detected'] = enzyme_consistency['n_always'] + enzyme_consistency['n_inconsistent']
enzyme_consistency = enzyme_consistency.sort_values('pct_inconsistent', ascending=False)

print(f"Enzyme consistency summary:")
print(f"  Mean inconsistent products per enzyme: {enzyme_consistency['n_inconsistent'].mean():.1f}")
print(f"  Median inconsistent products per enzyme: {enzyme_consistency['n_inconsistent'].median():.0f}")
print(f"  Max: {enzyme_consistency['n_inconsistent'].max()} (enzyme {enzyme_consistency.iloc[0]['Code']})")
print(f"  Min: {enzyme_consistency['n_inconsistent'].min()}")

print(f"\nTop 15 most inconsistent enzymes:")
print(enzyme_consistency[['Code', 'n_always', 'n_inconsistent', 'n_never', 'pct_inconsistent']].head(15).to_string(index=False))

In [ ]:
# Figure: Distribution of per-enzyme inconsistency rate
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: histogram of n_inconsistent per enzyme
ax = axes[0]
ax.hist(enzyme_consistency['n_inconsistent'], bins=20, color='steelblue', edgecolor='black', linewidth=0.5)
ax.axvline(enzyme_consistency['n_inconsistent'].mean(), color='red', linestyle='--', label=f"Mean: {enzyme_consistency['n_inconsistent'].mean():.1f}")
ax.set_xlabel('Number of Inconsistent Products', fontsize=12)
ax.set_ylabel('Number of Enzymes', fontsize=12)
ax.set_title('Per-Enzyme Inconsistency', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.2)

# Right: scatter of n_always_detected vs n_inconsistent
ax = axes[1]
ax.scatter(enzyme_consistency['n_always'], enzyme_consistency['n_inconsistent'],
           alpha=0.5, s=30, color='steelblue', edgecolors='black', linewidth=0.3)
ax.set_xlabel('Products Always Detected (3/3)', fontsize=12)
ax.set_ylabel('Products Inconsistently Detected', fontsize=12)
ax.set_title('Enzyme Activity Breadth vs Noise', fontsize=13)
ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "enzyme_consistency.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved: enzyme_consistency.png")

## Section 6: Bile Acid Core Consistency

In [ ]:
# Detection consistency by bile acid core (hydroxyl pattern)
hydroxyl_detection = replicate_stats.groupby('hydroxyl').agg(
    n_combos=('detection', 'count'),
    n_never=('detection', lambda x: (x == 'never_detected').sum()),
    n_always=('detection', lambda x: (x == 'always_detected').sum()),
    n_inconsistent=('detection', lambda x: (x == 'inconsistent').sum()),
).reset_index()

hydroxyl_detection['pct_inconsistent'] = hydroxyl_detection['n_inconsistent'] / hydroxyl_detection['n_combos'] * 100
hydroxyl_detection['pct_always'] = hydroxyl_detection['n_always'] / hydroxyl_detection['n_combos'] * 100
hydroxyl_detection = hydroxyl_detection.sort_values('pct_inconsistent', ascending=False)

print("Detection consistency by bile acid core:")
print(hydroxyl_detection[['hydroxyl', 'n_combos', 'pct_always', 'pct_inconsistent']].to_string(index=False))

In [ ]:
# Heatmap: inconsistency rate per amine x hydroxyl
amine_hyd_incons = replicate_stats.groupby(['amine', 'hydroxyl']).agg(
    n_combos=('detection', 'count'),
    n_inconsistent=('detection', lambda x: (x == 'inconsistent').sum()),
    n_always=('detection', lambda x: (x == 'always_detected').sum()),
).reset_index()

amine_hyd_incons['pct_inconsistent'] = amine_hyd_incons['n_inconsistent'] / amine_hyd_incons['n_combos'] * 100

# Pivot
incons_heatmap = amine_hyd_incons.pivot_table(
    index='amine', columns='hydroxyl', values='pct_inconsistent', fill_value=np.nan
)

# Sort by overall inconsistency
amine_overall_incons = amine_detection.set_index('amine')['pct_inconsistent']
order = amine_overall_incons.sort_values(ascending=False).index
incons_heatmap = incons_heatmap.reindex(order)

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(
    incons_heatmap,
    cmap='YlOrRd',
    vmin=0, vmax=50,
    annot=True, fmt='.0f',
    linewidths=0.5, linecolor='white',
    cbar_kws={'label': '% Inconsistent (detected in 1 or 2 of 3 reps)'},
    ax=ax,
    mask=incons_heatmap.isna(),
)

ax.set_xlabel('Bile Acid Core (Hydroxyl Pattern)', fontsize=12)
ax.set_ylabel('Amine', fontsize=12)
ax.set_title('Replicate Inconsistency Rate by Amine and Bile Acid Core\n(% of enzyme-product combos where detection varies across 3 replicates)', fontsize=13)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "inconsistency_amine_hydroxyl_heatmap.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved: inconsistency_amine_hydroxyl_heatmap.png")

## Section 7: Amine Activity Heatmap (Consensus)

In [ ]:
# Aggregate to amine level: for each (amine, hydroxyl), 
# what fraction of enzymes consistently detect it (3/3 reps)?

amine_hyd_activity = replicate_stats.groupby(['amine', 'hydroxyl']).agg(
    n_enzymes=('Code', 'nunique'),
    n_always_detected=('detection', lambda x: (x == 'always_detected').sum()),
    n_any_detected=('detection', lambda x: (x != 'never_detected').sum()),
).reset_index()

amine_hyd_activity['activity_rate_strict'] = amine_hyd_activity['n_always_detected'] / amine_hyd_activity['n_enzymes']
amine_hyd_activity['activity_rate_any'] = amine_hyd_activity['n_any_detected'] / amine_hyd_activity['n_enzymes']

# Pivot for heatmap (strict = 3/3 replicates)
activity_strict = amine_hyd_activity.pivot_table(
    index='amine', columns='hydroxyl', values='activity_rate_strict', fill_value=np.nan
)

# Order by overall activity
overall = amine_hyd_activity.groupby('amine')['n_always_detected'].sum() / amine_hyd_activity.groupby('amine')['n_enzymes'].sum()
order = overall.sort_values(ascending=False).index
activity_strict = activity_strict.reindex(order)

fig, ax = plt.subplots(figsize=(12, 12))
sns.heatmap(
    activity_strict,
    cmap='YlOrRd',
    vmin=0, vmax=0.7,
    annot=True, fmt='.2f',
    linewidths=0.5, linecolor='white',
    cbar_kws={'label': 'Activity Rate (strict: detected in 3/3 replicates)'},
    ax=ax,
    mask=activity_strict.isna(),
)

ax.set_xlabel('Bile Acid Core (Hydroxyl Pattern)', fontsize=12)
ax.set_ylabel('Amine', fontsize=12)
ax.set_title('Strict Activity Rate by Amine and Bile Acid Core\n(fraction of enzymes where product detected in ALL 3 replicates)', fontsize=13)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "amine_bile_acid_heatmap_strict.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved: amine_bile_acid_heatmap_strict.png")

## Section 8: Summary & Save

In [ ]:
# Comprehensive amine summary table
amine_summary = amine_detection.copy()

# Add CV info
cv_merge = amine_cv[['amine', 'median_cv', 'n_detected_combos']].copy()
amine_summary = amine_summary.merge(cv_merge, on='amine', how='left')

# Add overall activity rate (strict)
strict_activity = amine_hyd_activity.groupby('amine').agg(
    total_enzymes=('n_enzymes', 'sum'),
    total_always=('n_always_detected', 'sum'),
    total_any=('n_any_detected', 'sum'),
).reset_index()
strict_activity['strict_activity_rate'] = strict_activity['total_always'] / strict_activity['total_enzymes']
strict_activity['any_activity_rate'] = strict_activity['total_any'] / strict_activity['total_enzymes']
amine_summary = amine_summary.merge(strict_activity[['amine', 'strict_activity_rate', 'any_activity_rate']], on='amine', how='left')

# Sort by strict activity rate
amine_summary = amine_summary.sort_values('strict_activity_rate', ascending=False)

print("Amine Summary (sorted by strict activity rate):")
cols_to_show = ['amine', 'n_combos', 'pct_always', 'pct_inconsistent', 'pct_never', 
                'strict_activity_rate', 'any_activity_rate', 'median_cv']
print(amine_summary[cols_to_show].to_string(index=False))

In [ ]:
# Save outputs
amine_summary.to_csv(RESULTS_DIR / "amine_consistency_summary.csv", index=False)
amine_detection.to_csv(RESULTS_DIR / "amine_detection_consistency.csv", index=False)
enzyme_consistency.to_csv(RESULTS_DIR / "enzyme_consistency.csv", index=False)
amine_cv.to_csv(RESULTS_DIR / "amine_intensity_cv.csv", index=False)
hydroxyl_detection.to_csv(RESULTS_DIR / "hydroxyl_detection_consistency.csv", index=False)

print("Saved CSVs:")
for f in sorted(RESULTS_DIR.glob("*.csv")):
    print(f"  {f.name}")
print("\nSaved PNGs:")
for f in sorted(RESULTS_DIR.glob("*.png")):
    print(f"  {f.name}")

In [ ]:
# Final summary
print("=" * 80)
print("AMINE REPLICATE CONSISTENCY -- SUMMARY")
print("=" * 80)

print(f"\nData: {df_raw['Code'].nunique()} enzymes x 3 replicates x {len(product_info)} products")
print(f"Products: {len(product_info)} total ({df_subs.shape[1]-3} substrates + {df_amines.shape[1]-3} amines)")
print(f"Covering {len(unique_amines)} amines across {len(unique_hydroxyls)} bile acid cores")

n_total = len(replicate_stats)
n_never = (replicate_stats['detection'] == 'never_detected').sum()
n_always = (replicate_stats['detection'] == 'always_detected').sum()
n_incons = (replicate_stats['detection'] == 'inconsistent').sum()

print(f"\n--- Detection Consistency ---")
print(f"  Never detected (0/3):    {n_never:5d} ({n_never/n_total:.1%})")
print(f"  Always detected (3/3):   {n_always:5d} ({n_always/n_total:.1%})")
print(f"  Inconsistent (1-2/3):    {n_incons:5d} ({n_incons/n_total:.1%})")

print(f"\n--- Most Reliable Amines (lowest % inconsistent) ---")
top_reliable = amine_detection.sort_values('pct_inconsistent').head(5)
for _, row in top_reliable.iterrows():
    print(f"  {row['amine']:30s} {row['pct_inconsistent']:.1f}% inconsistent")

print(f"\n--- Noisiest Amines (highest % inconsistent) ---")
top_noisy = amine_detection.sort_values('pct_inconsistent', ascending=False).head(5)
for _, row in top_noisy.iterrows():
    print(f"  {row['amine']:30s} {row['pct_inconsistent']:.1f}% inconsistent")

print(f"\n--- Implications ---")
print(f"1. {n_incons/n_total:.1%} of all enzyme-product combinations show inconsistent detection")
print(f"   across replicates, representing a measurement noise floor")
print(f"2. Amines with high inconsistency contribute unreliable training labels")
print(f"3. The noise ceiling limits achievable model performance")

print("\nDone!")